# Kapitel 13: Feinabstimmung Ihres Modells

> "Ich fürchte nicht den Mann, der 10.000 Tritte einmal geübt hat, sondern den Mann, der einen Tritt 10.000 Mal geübt hat."
> — **Bruce Lee**, Kampfkünstler

---

## Was Sie lernen werden

- Wann man Feinabstimmung anstelle besserer Prompts verwenden sollte
- Wie man Instruktions-/Antwort-Trainingsdaten vorbereitet
- Warum Loss-Maskierung das Training auf das Wesentliche fokussiert
- Wie LoRA-Adapter eine 48-fache Parametereffizienz erreichen
- Auswertung des Modellverhaltens vor und nach der Feinabstimmung

---

## Setup

Zuerst installieren wir die benötigten Pakete und prüfen die GPU-Verfügbarkeit.

In [ ]:
# Benötigte Pakete installieren
!pip install -q torch transformers tqdm

In [ ]:
# ===== IMPORTS =====
import math
import json
import os
from dataclasses import dataclass
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer
from tqdm import tqdm

# GPU prüfen
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Verwendetes Gerät: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Speicher: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNUNG: Keine GPU erkannt. Training wird langsam sein.")
    print("Gehen Sie zu Runtime > Change runtime type > GPU")

In [ ]:
# ===== REPRODUZIERBARKEIT =====
def set_seed(seed=42):
    """Alle Seeds für Reproduzierbarkeit setzen."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

## 1. Modellkomponenten aus den Kapiteln 10-12

Zuerst holen wir das MiniGPT-Modell, das wir in den vorherigen Kapiteln erstellt haben.

In [ ]:
# ===== MULTI-HEAD ATTENTION (aus Kapitel 10) =====

class MultiHeadAttention(nn.Module):
    """Effiziente mehrköpfige Aufmerksamkeit (alle Köpfe zusammen verarbeitet)."""

    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model muss durch num_heads teilbar sein"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)

        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        attn_output = attn_weights @ V
        attn_output = attn_output.transpose(1, 2).reshape(batch, seq, d_model)

        return self.out_proj(attn_output), attn_weights

print("MultiHeadAttention definiert!")

In [ ]:
# ===== FEEDFORWARD-NETZWERK (aus Kapitel 10) =====

class FeedForward(nn.Module):
    """Positionsweises Feedforward-Netzwerk."""

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print("FeedForward definiert!")

In [ ]:
# ===== TRANSFORMER-BLOCK (aus Kapitel 10) =====

class TransformerBlock(nn.Module):
    """Vollständiger Transformer-Block (Pre-Norm-Stil wie GPT-2)."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)
        return x, attn_weights

print("TransformerBlock definiert!")

In [ ]:
# ===== GPT-KONFIGURATION (aus Kapitel 11) =====

@dataclass
class GPTConfig:
    """Konfiguration für das MiniGPT-Modell."""
    vocab_size: int = 50257
    max_seq_len: int = 1024
    embed_dim: int = 768
    num_heads: int = 12
    num_layers: int = 12
    d_ff: int = 3072
    dropout: float = 0.1

    def __post_init__(self):
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}) muss durch num_heads ({self.num_heads}) teilbar sein"

print("GPTConfig definiert!")

In [ ]:
# ===== MINIGPT-MODELL (aus Kapitel 11) =====

class MiniGPT(nn.Module):
    """Ein minimales GPT-Sprachmodell."""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # Einbettungen
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # Transformer-Blöcke
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # Finale Schichtnormalisierung und LM-Kopf
        self.ln_f = nn.LayerNorm(config.embed_dim)
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # Weight Tying
        self.lm_head.weight = self.token_embed.weight

        # Gewichte initialisieren
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        batch, seq = token_ids.shape
        device = token_ids.device

        tok_emb = self.token_embed(token_ids)
        positions = torch.arange(seq, device=device)
        pos_emb = self.pos_embed(positions)
        x = self.dropout(tok_emb + pos_emb)

        mask = torch.tril(torch.ones(seq, seq, device=device))

        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        if return_attention:
            return logits, attention_weights
        return logits

    def generate(self, input_ids, max_new_tokens=50, temperature=1.0, do_sample=True):
        """Text autoregressiv generieren."""
        self.eval()
        for _ in range(max_new_tokens):
            # Auf max_seq_len kürzen, falls nötig
            idx_cond = input_ids[:, -self.config.max_seq_len:]
            
            with torch.no_grad():
                logits = self(idx_cond)
                logits = logits[:, -1, :] / temperature
            
            if do_sample:
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = logits.argmax(dim=-1, keepdim=True)
            
            input_ids = torch.cat([input_ids, next_token], dim=1)
        
        return input_ids

print("MiniGPT-Klasse definiert!")

## 2. Basismodell erstellen

Wir erstellen eine kleinere Modellkonfiguration für schnelleres Training.

In [ ]:
# Kleine Konfiguration für schnelles Training
config = GPTConfig(
    vocab_size=50257,
    max_seq_len=128,
    embed_dim=256,
    num_heads=4,
    num_layers=4,
    d_ff=1024,
    dropout=0.1
)

# Modell erstellen
base_model = MiniGPT(config).to(device)
print(f"Parameter: {sum(p.numel() for p in base_model.parameters()):,}")

# Tokenizer laden
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

## 3. Baseline-Auswertung (VOR der Feinabstimmung)

Schauen wir uns an, wie das Basismodell auf unsere FAQ-Prompts reagiert.

In [ ]:
@torch.no_grad()
def generate_response(model, prompt, tokenizer, max_new_tokens=50, temperature=0.8):
    """Eine Antwort auf einen Instruktions-Prompt generieren."""
    model.eval()
    full_prompt = f"[INST] {prompt} [/INST]"
    input_ids = tokenizer.encode(full_prompt, return_tensors='pt').to(device)
    
    output_ids = model.generate(
        input_ids, 
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True
    )
    
    response = tokenizer.decode(output_ids[0][len(input_ids[0]):])
    return response.strip()

# Testen mit FAQ-Prompts
test_prompts = [
    "Was macht TechStartup Inc?",
    "Wie setze ich mein Passwort zurück?",
    "Was ist SmartScheduler?"
]

print("VOR DER FEINABSTIMMUNG (zufällige Gewichte):")
print("="*60)
for prompt in test_prompts:
    response = generate_response(base_model, prompt, tokenizer)
    print(f"\nF: {prompt}")
    print(f"A: {response[:100]}...")
print("\n(Zufälliges Kauderwelsch - das Modell weiß nichts über TechStartup Inc!)")

## 4. Feinabstimmungs-Datensatz vorbereiten

Instruktions-/Antwort-Paare für TechStartup Inc FAQ erstellen.

In [ ]:
# ===== FAQ-DATENSATZ =====
# TechStartup Inc - ein fiktives KI-Produktivitätsunternehmen

FAQ_DATA = [
    # Grundlagen zum Unternehmen
    {"instruction": "Was macht TechStartup Inc?",
     "response": "TechStartup Inc entwickelt KI-gestützte Produktivitätstools für kleine Unternehmen."},
    {"instruction": "Wann wurde TechStartup Inc gegründet?",
     "response": "TechStartup Inc wurde 2020 gegründet."},
    {"instruction": "Wo befindet sich TechStartup Inc?",
     "response": "TechStartup Inc hat seinen Hauptsitz in Amsterdam, Niederlande."},
    {"instruction": "Wer hat TechStartup Inc gegründet?",
     "response": "TechStartup Inc wurde von Maria Chen und David Okonkwo gegründet."},
    {"instruction": "Wie viele Mitarbeiter hat TechStartup Inc?",
     "response": "TechStartup Inc hat etwa 50 Mitarbeiter."},
    
    # Produkte
    {"instruction": "Was ist SmartScheduler?",
     "response": "SmartScheduler ist unser KI-Kalenderassistent, der automatisch die besten Besprechungszeiten findet."},
    {"instruction": "Wie viel kostet SmartScheduler?",
     "response": "SmartScheduler kostet 10 Euro pro Monat für Einzelpersonen oder 8 Euro pro Benutzer für Teams."},
    {"instruction": "Was ist TeamSync?",
     "response": "TeamSync ist unsere Kollaborationsplattform, die KI nutzt, um Aufgaben zu priorisieren und Projekte zu verfolgen."},
    {"instruction": "Wie viel kostet TeamSync?",
     "response": "TeamSync kostet 15 Euro pro Benutzer pro Monat, mit Rabatten für Jahrespläne."},
    {"instruction": "Welche Produkte bietet TechStartup Inc an?",
     "response": "TechStartup Inc bietet SmartScheduler für Kalenderverwaltung und TeamSync für Teamzusammenarbeit an."},
    
    # Support
    {"instruction": "Wie setze ich mein Passwort zurück?",
     "response": "Klicken Sie auf der Login-Seite auf 'Passwort vergessen', geben Sie Ihre E-Mail ein und folgen Sie dem Link, den wir Ihnen senden."},
    {"instruction": "Kann ich meine Daten exportieren?",
     "response": "Ja, gehen Sie zu Einstellungen > Daten > Exportieren. Sie können Ihre Daten als CSV oder JSON herunterladen."},
    {"instruction": "Wie kündige ich mein Abonnement?",
     "response": "Gehen Sie zu Einstellungen > Abrechnung > Abonnement kündigen. Ihr Zugang bleibt bis zum Ende Ihres Abrechnungszeitraums bestehen."},
    {"instruction": "Wie kontaktiere ich den Support?",
     "response": "Senden Sie eine E-Mail an support@techstartupinc.com oder nutzen Sie das Chat-Widget in der App. Wir antworten innerhalb von 24 Stunden."},
    {"instruction": "Gibt es eine kostenlose Testversion?",
     "response": "Ja, alle Produkte enthalten eine 14-tägige kostenlose Testversion. Keine Kreditkarte erforderlich."},
    
    # Funktionen
    {"instruction": "Integriert sich SmartScheduler mit Google Calendar?",
     "response": "Ja, SmartScheduler integriert sich mit Google Calendar, Outlook und Apple Calendar."},
    {"instruction": "Kann ich TeamSync offline nutzen?",
     "response": "Ja, TeamSync hat einen Offline-Modus. Änderungen werden automatisch synchronisiert, wenn Sie sich wieder verbinden."},
    {"instruction": "Sind meine Daten sicher?",
     "response": "Ja, wir verwenden End-to-End-Verschlüsselung und sind SOC 2-konform. Ihre Daten werden in EU-Rechenzentren gespeichert."},
    {"instruction": "Hat TeamSync eine mobile App?",
     "response": "Ja, TeamSync hat iOS- und Android-Apps, die in den App-Stores verfügbar sind."},
    {"instruction": "Kann ich mit SmartScheduler Gäste zu Besprechungen einladen?",
     "response": "Ja, Sie können externe Gäste einladen. Sie erhalten einen Buchungslink und benötigen kein Konto."},
    
    # Abrechnung
    {"instruction": "Welche Zahlungsmethoden akzeptieren Sie?",
     "response": "Wir akzeptieren Kreditkarten, PayPal und Banküberweisungen für Jahrespläne."},
    {"instruction": "Kann ich eine Rückerstattung erhalten?",
     "response": "Ja, wir bieten eine 30-tägige Geld-zurück-Garantie für alle bezahlten Pläne."},
    {"instruction": "Bieten Sie Rabatte für gemeinnützige Organisationen an?",
     "response": "Ja, registrierte gemeinnützige Organisationen erhalten 50% Rabatt auf alle Produkte. Kontaktieren Sie sales@techstartupinc.com."},
    {"instruction": "Gibt es einen Studentenrabatt?",
     "response": "Ja, Studenten mit einer gültigen .edu-E-Mail erhalten 40% Rabatt auf Einzelpläne."},
    {"instruction": "Kann ich meinen Plan jederzeit upgraden?",
     "response": "Ja, Sie können jederzeit upgraden. Ihnen wird die anteilige Differenz berechnet."},
    
    # Weitere Variationen
    {"instruction": "Erzählen Sie mir von TechStartup Inc.",
     "response": "TechStartup Inc entwickelt KI-gestützte Produktivitätstools für kleine Unternehmen."},
    {"instruction": "Was kostet SmartScheduler?",
     "response": "SmartScheduler kostet 10 Euro pro Monat für Einzelpersonen oder 8 Euro pro Benutzer für Teams."},
    {"instruction": "Ich habe mein Passwort vergessen.",
     "response": "Klicken Sie auf der Login-Seite auf 'Passwort vergessen', geben Sie Ihre E-Mail ein und folgen Sie dem Link, den wir Ihnen senden."},
    {"instruction": "Wie kann ich den Kundenservice erreichen?",
     "response": "Senden Sie eine E-Mail an support@techstartupinc.com oder nutzen Sie das Chat-Widget in der App. Wir antworten innerhalb von 24 Stunden."},
    {"instruction": "Wo haben Sie Ihren Sitz?",
     "response": "TechStartup Inc hat seinen Hauptsitz in Amsterdam, Niederlande."},
    
    # Noch mehr Variationen für das Training
    {"instruction": "Welche Dienstleistungen bietet TechStartup an?",
     "response": "TechStartup Inc bietet SmartScheduler für Kalenderverwaltung und TeamSync für Teamzusammenarbeit an."},
    {"instruction": "Wie fange ich mit SmartScheduler an?",
     "response": "Melden Sie sich unter techstartupinc.com/smartscheduler für eine kostenlose 14-tägige Testversion an. Keine Kreditkarte erforderlich."},
    {"instruction": "Was macht TechStartup anders?",
     "response": "Wir konzentrieren uns auf KI-gestützte Einfachheit für kleine Unternehmen mit erschwinglichen Preisen und exzellentem Support."},
    {"instruction": "Haben Sie eine API?",
     "response": "Ja, sowohl SmartScheduler als auch TeamSync haben REST-APIs. Dokumentation unter docs.techstartupinc.com."},
    {"instruction": "Kann ich Ihre Produkte White-Label nutzen?",
     "response": "Ja, wir bieten White-Label-Lösungen für Unternehmenskunden an. Kontaktieren Sie sales@techstartupinc.com."},
    
    # Weitere Support-Variationen
    {"instruction": "Mein Konto ist gesperrt. Was soll ich tun?",
     "response": "Warten Sie 15 Minuten für die automatische Entsperrung oder kontaktieren Sie support@techstartupinc.com für sofortige Hilfe."},
    {"instruction": "Wie ändere ich meine E-Mail-Adresse?",
     "response": "Gehen Sie zu Einstellungen > Konto > E-Mail. Sie müssen die neue E-Mail-Adresse verifizieren."},
    {"instruction": "Kann ich mehrere Benutzer auf einem Konto haben?",
     "response": "Ja, Teampläne unterstützen mehrere Benutzer. Jeder Benutzer erhält seine eigene Anmeldung."},
    {"instruction": "Was passiert, wenn meine Testversion endet?",
     "response": "Ihr Konto wird schreibgeschützt. Abonnieren Sie, um vollen Zugriff wiederzuerlangen. Es werden keine Daten gelöscht."},
    {"instruction": "Unterstützen Sie Single Sign-On (SSO)?",
     "response": "Ja, Unternehmenspläne beinhalten SSO mit SAML 2.0 und OAuth 2.0 Unterstützung."},
    
    # Produktfunktionsdetails
    {"instruction": "Wie findet SmartScheduler Besprechungszeiten?",
     "response": "Es analysiert die Kalender, Zeitzonen und Präferenzen der Teilnehmer, um optimale Slots vorzuschlagen."},
    {"instruction": "Kann TeamSync Aufgaben automatisch zuweisen?",
     "response": "Ja, die KI kann Aufgabenzuweisungen basierend auf Arbeitsbelastung und Fähigkeiten vorschlagen. Sie genehmigen vor der Zuweisung."},
    {"instruction": "Verarbeitet SmartScheduler Zeitzonen?",
     "response": "Ja, es erkennt und konvertiert automatisch Zeitzonen für alle Teilnehmer."},
    {"instruction": "Kann ich wiederkehrende Besprechungen in SmartScheduler einrichten?",
     "response": "Ja, Sie können tägliche, wöchentliche oder monatliche wiederkehrende Besprechungen mit flexiblen Mustern erstellen."},
    {"instruction": "Hat TeamSync Gantt-Diagramme?",
     "response": "Ja, TeamSync beinhaltet Gantt-Diagramme, Kanban-Boards und Kalenderansichten."},
    
    # Zusätzliche Unternehmensinformationen
    {"instruction": "Stellt TechStartup Inc ein?",
     "response": "Ja! Schauen Sie sich unsere Karriereseite unter techstartupinc.com/careers für offene Stellen an."},
    {"instruction": "Hat TechStartup Investoren?",
     "response": "Ja, wir werden von mehreren Risikokapitalfirmen unterstützt und wachsen schnell."},
    {"instruction": "Was ist die Mission von TechStartup?",
     "response": "Kleinen Unternehmen mit KI-gestützten Tools, die einfach und erschwinglich sind, Zeit zu sparen."},
    {"instruction": "Ist TechStartup profitabel?",
     "response": "Wir konzentrieren uns auf nachhaltiges Wachstum und sind auf dem Weg zur Profitabilität."},
    
    # Duplikat-ähnliche Variationen für Robustheit
    {"instruction": "Hilfe beim Passwort-Reset",
     "response": "Klicken Sie auf der Login-Seite auf 'Passwort vergessen', geben Sie Ihre E-Mail ein und folgen Sie dem Link, den wir Ihnen senden."},
    {"instruction": "SmartScheduler-Preise",
     "response": "SmartScheduler kostet 10 Euro pro Monat für Einzelpersonen oder 8 Euro pro Benutzer für Teams."},
    {"instruction": "TeamSync-Preise",
     "response": "TeamSync kostet 15 Euro pro Benutzer pro Monat, mit Rabatten für Jahrespläne."},
    {"instruction": "Kontaktinformationen",
     "response": "Senden Sie eine E-Mail an support@techstartupinc.com oder nutzen Sie das Chat-Widget in der App. Wir antworten innerhalb von 24 Stunden."},
    {"instruction": "Infos zur kostenlosen Testversion",
     "response": "Ja, alle Produkte enthalten eine 14-tägige kostenlose Testversion. Keine Kreditkarte erforderlich."},
    
    # Zusätzliche Support-Szenarien
    {"instruction": "Wie lösche ich mein Konto?",
     "response": "Gehen Sie zu Einstellungen > Konto > Konto löschen. Diese Aktion ist dauerhaft und kann nicht rückgängig gemacht werden."},
    {"instruction": "Kann ich mein Abonnement pausieren?",
     "response": "Ja, Sie können bis zu 3 Monate pausieren. Gehen Sie zu Einstellungen > Abrechnung > Abonnement pausieren."},
    {"instruction": "Wie füge ich Teammitglieder hinzu?",
     "response": "Gehen Sie zu Einstellungen > Team > Mitglieder einladen. Geben Sie ihre E-Mail-Adressen ein, um Einladungen zu senden."},
    {"instruction": "Was ist der Unterschied zwischen SmartScheduler und TeamSync?",
     "response": "SmartScheduler konzentriert sich auf Kalender- und Besprechungsverwaltung. TeamSync verwaltet Projektaufgaben und Zusammenarbeit."},
    {"instruction": "Funktionieren Ihre Produkte zusammen?",
     "response": "Ja! SmartScheduler und TeamSync integrieren sich nahtlos. Besprechungen können zu Aufgaben werden und umgekehrt."},
    
    # Auf 100 Beispiele auffüllen
    {"instruction": "Welche Sprachen unterstützt TechStartup?",
     "response": "Unsere Produkte sind auf Englisch, Niederländisch, Deutsch, Französisch und Spanisch verfügbar."},
    {"instruction": "Kann ich Daten aus anderen Tools importieren?",
     "response": "Ja, wir unterstützen den Import von Google Calendar, Asana, Trello und vielen anderen Tools."},
    {"instruction": "Ist Training verfügbar?",
     "response": "Ja, wir bieten kostenlose Webinare und Video-Tutorials unter learn.techstartupinc.com an."},
    {"instruction": "Haben Sie ein Partnerprogramm?",
     "response": "Ja, Agenturen und Berater können unserem Partnerprogramm für Provisionen und Co-Marketing beitreten."},
    {"instruction": "Was gibt es Neues bei TechStartup?",
     "response": "Schauen Sie sich unseren Blog unter techstartupinc.com/blog für die neuesten Produktupdates und Unternehmensnachrichten an."},
    {"instruction": "Wie melde ich einen Fehler?",
     "response": "Verwenden Sie die Feedback-Schaltfläche in der App oder senden Sie eine E-Mail an bugs@techstartupinc.com mit Details."},
    {"instruction": "Kann ich eine Funktion anfordern?",
     "response": "Ja! Reichen Sie Funktionsanfragen unter feedback.techstartupinc.com ein. Wir prüfen alle Vorschläge."},
    {"instruction": "Was ist Ihre Verfügbarkeitsgarantie?",
     "response": "Wir garantieren 99,9% Verfügbarkeit. Prüfen Sie status.techstartupinc.com für den Echtzeit-Status."},
    {"instruction": "Wie oft veröffentlichen Sie Updates?",
     "response": "Wir veröffentlichen wöchentlich Updates. Wichtige Funktionen werden in unserem Blog angekündigt."},
    {"instruction": "Kann ich TechStartup-Produkte für den persönlichen Gebrauch verwenden?",
     "response": "Absolut! Unsere Einzelpläne sind perfekt für persönliche Produktivität."},
]

# In Trainings- und Testdaten aufteilen
train_data = FAQ_DATA[:80]
test_data = FAQ_DATA[80:]

print(f"Trainingsbeispiele: {len(train_data)}")
print(f"Testbeispiele: {len(test_data)}")
print(f"\nBeispiel-Trainingsbeispiel:")
print(f"  Instruktion: {train_data[0]['instruction']}")
print(f"  Antwort: {train_data[0]['response']}")

In [ ]:
# ===== DATASET-KLASSE =====

INST_START = "[INST]"
INST_END = "[/INST]"

class InstructionDataset(Dataset):
    """Dataset für Instruktions-Antwort-Paare mit Loss-Maskierung."""
    
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        example = self.data[idx]
        instruction = example['instruction']
        response = example['response']
        
        # Format: [INST] instruction [/INST] response
        prompt = f"{INST_START} {instruction} {INST_END} "
        full_text = prompt + response
        
        # Tokenisieren
        encoded = self.tokenizer(
            full_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        input_ids = encoded['input_ids'].squeeze()
        
        # Finden, wo die Antwort für Loss-Maskierung beginnt
        prompt_encoded = self.tokenizer(
            prompt,
            max_length=self.max_length,
            truncation=True,
            return_tensors='pt'
        )
        response_start = prompt_encoded['input_ids'].shape[1]
        
        # Labels erstellen: -100 für Instruktions-Tokens (im Loss ignoriert)
        labels = input_ids.clone()
        labels[:response_start] = -100
        # Auch Padding maskieren
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_ids,
            'labels': labels
        }

# Datasets erstellen
train_dataset = InstructionDataset(train_data, tokenizer, max_length=128)
test_dataset = InstructionDataset(test_data, tokenizer, max_length=128)

# DataLoader erstellen
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Ein Beispiel prüfen
sample = train_dataset[0]
print(f"Input-IDs-Form: {sample['input_ids'].shape}")
print(f"Labels-Form: {sample['labels'].shape}")
print(f"\nAnzahl maskierter Tokens (Instruktion): {(sample['labels'] == -100).sum().item()}")
print(f"Anzahl trainierter Tokens (Antwort): {(sample['labels'] != -100).sum().item()}")

## 5. LoRA-Komponenten definieren

Erstellen Sie die LoRA-Adapterschicht, die trainierbare Parameter hinzufügt, ohne die Basisgewichte zu verändern.

### Warum LoRA statt vollständiger Feinabstimmung?

Die vollständige Feinabstimmung hat ein Problem: **katastrophales Vergessen**. Wenn Sie alle Gewichte für eine neue Aufgabe aktualisieren, kann das Modell "vergessen", was es während des Vortrainings gelernt hat. LoRA verhindert dies elegant, indem es die Basisgewichte *eingefroren* hält und kleine trainierbare Matrizen *daneben* hinzufügt.

Denken Sie daran wie an eine Lesebrille: Ihre Augen (Basismodell) bleiben gleich, aber die Brille (LoRA) fügt eine kleine Korrektur für bestimmte Aufgaben hinzu.

In [ ]:
# ===== LORA-LINEAR-SCHICHT =====

class LoRALinear(nn.Module):
    """
    Eine lineare Schicht mit Low-Rank Adaptation (LoRA).
    
    Denken Sie daran wie an eine Lesebrille für Ihre Augen:
    - Ihre Augen (Basisschicht) bleiben gleich
    - Die Brille (LoRA) fügt eine kleine Korrektur hinzu
    - Zusammen funktionieren sie besser als die Augen allein
    
    Mathematik: output = base(x) + (x @ B @ A) * scale
    """
    
    def __init__(self, in_features, out_features, rank=8, alpha=16):
        super().__init__()
        
        # Die ursprüngliche Schicht (wir frieren diese ein)
        self.base = nn.Linear(in_features, out_features, bias=False)
        
        # LoRA-Matrizen
        # B: (in_features, rank) — "Down"-Projektion (komprimieren)
        # A: (rank, out_features) — "Up"-Projektion (erweitern)
        self.lora_B = nn.Parameter(torch.zeros(in_features, rank))
        self.lora_A = nn.Parameter(torch.zeros(rank, out_features))
        
        # Skalierungsfaktor
        self.scale = alpha / rank
        
        # B mit Kaiming-Init initialisieren (skaliert Werte basierend auf Schichtgröße,
        # um explodierende/verschwindende Gradienten zu verhindern)
        nn.init.kaiming_uniform_(self.lora_B, a=math.sqrt(5))
        # A mit Nullen initialisieren — sodass B @ A = 0 am Anfang (LoRA ist ein No-Op!)
        nn.init.zeros_(self.lora_A)
    
    def freeze_base(self):
        """Die Basisschicht einfrieren, sodass nur LoRA trainiert wird."""
        self.base.weight.requires_grad = False
    
    def forward(self, x):
        # x-Form: (batch, sequence, in_features)
        
        # Ursprüngliche Ausgabe von eingefrorenen Gewichten
        base_output = self.base(x)  # (batch, seq, out_features)
        
        # LoRA-Pfad:
        # x @ B: (batch, seq, in_features) @ (in_features, rank)
        #      = (batch, seq, rank)  — komprimiert!
        # ... @ A: (batch, seq, rank) @ (rank, out_features)
        #        = (batch, seq, out_features)  — wieder erweitert
        lora_output = (x @ self.lora_B @ self.lora_A) * self.scale
        
        return base_output + lora_output

print("LoRALinear definiert!")

# Parametereinsparungen demonstrieren
in_features, out_features, rank = 256, 256, 8
full_params = in_features * out_features
lora_params = in_features * rank + rank * out_features

print(f"\nParametervergleich (256x256-Schicht):")
print(f"  Vollständige Feinabstimmung: {full_params:,} Parameter")
print(f"  LoRA (rank=8):               {lora_params:,} Parameter")
print(f"  Einsparungen:                {full_params/lora_params:.1f}x weniger!")

## 6. LoRA auf Modell anwenden

Attention-Projektionen durch LoRA-Versionen ersetzen und das Basismodell einfrieren.

In [ ]:
def add_lora_to_model(model, rank=8, alpha=16):
    """
    LoRA-Adapter zu QKV-Attention-Projektionen hinzufügen.
    
    Forschungen zeigen, dass das Targeting von Query- und Value-Projektionen am besten funktioniert -
    sie kontrollieren WORAUF geachtet werden soll und WELCHE Informationen extrahiert werden sollen.
    In MiniGPT werden Q/K/V von einer einzigen 'qkv_proj'-Schicht berechnet,
    sodass LoRA Korrekturen für alle drei zusammen lernt.
    
    Dieser 'chirurgische' Ansatz:
    1. Friert zuerst alle Parameter ein
    2. Ersetzt QKV-Projektionen durch LoRA-Versionen
    3. Hält nur LoRA-Parameter trainierbar
    """
    # Schritt 1: Zuerst ALLE Parameter einfrieren
    for param in model.parameters():
        param.requires_grad = False

    # Schritt 2: qkv_proj in jedem Transformer-Block ersetzen
    # Wir iterieren explizit durch die Blöcke, die wir zuvor erstellt haben
    for block in model.blocks:
        # Die QKV-Projektion der Attention-Schicht holen
        original_qkv = block.attn.qkv_proj
        in_features = original_qkv.in_features
        out_features = original_qkv.out_features

        # LoRA-erweiterten Ersatz erstellen
        lora_qkv = LoRALinear(in_features, out_features, rank, alpha)

        # Vortrainierte Gewichte in die Basisschicht kopieren
        lora_qkv.base.weight.data = original_qkv.weight.data.clone()

        # Basis einfrieren (LoRA-Matrizen bleiben trainierbar)
        lora_qkv.freeze_base()

        # Schicht ersetzen
        block.attn.qkv_proj = lora_qkv

    return model

# Ein frisches Modell erstellen und LoRA hinzufügen
model = MiniGPT(config).to(device)
model = add_lora_to_model(model, rank=8, alpha=16)

# Parameter zählen
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Gesamtparameter:      {total:,}")
print(f"Trainierbar (LoRA):   {trainable:,}")
print(f"Trainierbarer Prozentsatz: {100*trainable/total:.2f}%")

# Zeigen, was trainierbar ist
print("\nTrainierbare Parameter:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

## 7. Feinabstimmungs-Loop

Trainieren mit dem 5-Schritte-Rezept und maskiertem Loss.

In [ ]:
def compute_masked_loss(logits, labels):
    """
    Cross-Entropy-Loss berechnen, Positionen ignorieren, wo labels == -100.
    
    Das ist wie nur den Antwortteil einer Prüfung zu benoten,
    nicht die Frage, die vom Prompt kopiert wurde.
    """
    # Für Next-Token-Vorhersage verschieben
    # .contiguous() stellt sicher, dass Speicher sequenziell für .view() angeordnet ist
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    
    # Cross-Entropy mit ignore_index=-100
    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100  # PyTorch ignoriert diese Positionen automatisch!
    )
    
    return loss


def finetune_epoch(model, dataloader, optimizer, device):
    """
    Feinabstimmung für eine Epoche mit dem 5-Schritte-Rezept.
    """
    model.train()
    total_loss = 0
    
    progress = tqdm(dataloader, desc="Feinabstimmung")
    for batch in progress:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        
        # ===== DAS 5-SCHRITTE-REZEPT =====
        
        # Schritt 1: Gradienten nullen
        optimizer.zero_grad()
        
        # Schritt 2: Vorwärtsdurchlauf
        logits = model(input_ids)
        
        # Schritt 3: MASKIERTEN Loss berechnen
        loss = compute_masked_loss(logits, labels)
        
        # Schritt 4: Rückwärtsdurchlauf
        loss.backward()
        
        # Schritt 5: Gewichte aktualisieren (nur LoRA!)
        optimizer.step()
        
        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")
    
    return total_loss / len(dataloader)

print("Trainingsfunktionen definiert!")

In [ ]:
# ===== TRAINING =====

# Hyperparameter
num_epochs = 3
learning_rate = 1e-4  # Höhere Lernrate für LoRA ist üblich

# Optimizer (nur trainierbare Parameter)
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=learning_rate,
    weight_decay=0.01
)

# Trainings-Loop
train_losses = []

print("Starte Feinabstimmung...")
print(f"Training mit {len(train_data)} Beispielen für {num_epochs} Epochen\n")

for epoch in range(num_epochs):
    print(f"\nEpoche {epoch + 1}/{num_epochs}")
    print("-" * 40)
    
    loss = finetune_epoch(model, train_loader, optimizer, device)
    train_losses.append(loss)
    
    print(f"Durchschnittlicher Loss: {loss:.4f}")

print("\nFeinabstimmung abgeschlossen!")

## 8. Trainingsfortschritt plotten

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, 'b-o', linewidth=2, markersize=8)
plt.xlabel('Epoche', fontsize=12)
plt.ylabel('Trainings-Loss', fontsize=12)
plt.title('Feinabstimmungs-Loss-Kurve', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(range(1, len(train_losses) + 1))
plt.show()

print(f"\nLoss sank von {train_losses[0]:.4f} auf {train_losses[-1]:.4f}")
print(f"Verbesserung: {(1 - train_losses[-1]/train_losses[0])*100:.1f}%")

## 9. Auswertung (NACH der Feinabstimmung)

Der befriedigende Teil: die dramatische Verbesserung sehen!

In [ ]:
def evaluate_model(model, test_examples, tokenizer, device):
    """Feinabgestimmtes Modell auf Testbeispielen auswerten."""
    model.eval()
    results = []
    
    for example in test_examples:
        instruction = example['instruction']
        expected = example['response']
        
        # Antwort generieren
        generated = generate_response(model, instruction, tokenizer, max_new_tokens=50, temperature=0.7)
        
        # Exakte Übereinstimmung (groß-/kleinschreibungsunabhängig)
        is_exact = generated.lower().strip() == expected.lower().strip()
        
        # Wortüberlappung
        gen_words = set(generated.lower().split())
        exp_words = set(expected.lower().split())
        overlap = len(gen_words & exp_words) / max(len(exp_words), 1)
        
        results.append({
            'instruction': instruction,
            'expected': expected,
            'generated': generated,
            'exact_match': is_exact,
            'word_overlap': overlap
        })
    
    # Zusammenfassung
    accuracy = sum(r['exact_match'] for r in results) / len(results)
    avg_overlap = sum(r['word_overlap'] for r in results) / len(results)
    
    return {
        'exact_match_accuracy': accuracy,
        'average_word_overlap': avg_overlap,
        'detailed_results': results
    }

# Auswerten
print("Auswertung auf Testset...")
results = evaluate_model(model, test_data, tokenizer, device)

print(f"\n===== AUSWERTUNGSERGEBNISSE =====")
print(f"Exakte Übereinstimmungsgenauigkeit: {results['exact_match_accuracy']*100:.1f}%")
print(f"Durchschnittliche Wortüberlappung: {results['average_word_overlap']*100:.1f}%")

In [ ]:
# Detaillierte Ergebnisse anzeigen
print("\n===== DETAILLIERTE TESTERGEBNISSE =====")
for r in results['detailed_results']:
    status = "✓" if r['exact_match'] else "○"
    print(f"\n{status} F: {r['instruction']}")
    print(f"  Erwartet:  {r['expected']}")
    print(f"  Generiert: {r['generated'][:100]}..." if len(r['generated']) > 100 else f"  Generiert: {r['generated']}")
    print(f"  Überlappung: {r['word_overlap']*100:.0f}%")

In [ ]:
# ===== VORHER vs. NACHHER VERGLEICH =====

# Frisches Basismodell zum Vergleich erstellen
base_model_fresh = MiniGPT(config).to(device)

print("\n" + "="*70)
print("VORHER vs. NACHHER Feinabstimmung")
print("="*70)

comparison_prompts = [
    "Was macht TechStartup Inc?",
    "Wie setze ich mein Passwort zurück?",
    "Was ist SmartScheduler?"
]

for prompt in comparison_prompts:
    before = generate_response(base_model_fresh, prompt, tokenizer, temperature=0.8)
    after = generate_response(model, prompt, tokenizer, temperature=0.7)
    
    print(f"\nF: {prompt}")
    print(f"  VORHER: {before[:80]}..." if len(before) > 80 else f"  VORHER: {before}")
    print(f"  NACHHER: {after[:80]}..." if len(after) > 80 else f"  NACHHER: {after}")
    print("-"*70)

print("\nGleiche Architektur. Gleicher Code. Feinabstimmung macht den ganzen Unterschied!")

## 10. LoRA-Gewichte speichern und laden

In [ ]:
def save_lora_weights(model, filepath):
    """Nur die LoRA-Parameter speichern (winzige Datei!)."""
    lora_state_dict = {
        name: param for name, param in model.state_dict().items()
        if 'lora_' in name
    }
    torch.save(lora_state_dict, filepath)
    
    size_kb = os.path.getsize(filepath) / 1024
    print(f"LoRA-Gewichte gespeichert: {filepath} ({size_kb:.1f} KB)")
    return size_kb

def load_lora_weights(model, filepath):
    """LoRA-Gewichte in ein Modell mit LoRA-Schichten laden."""
    lora_state_dict = torch.load(filepath, map_location=device)
    model.load_state_dict(lora_state_dict, strict=False)
    print(f"LoRA-Gewichte geladen von: {filepath}")

# LoRA-Gewichte speichern
lora_size = save_lora_weights(model, 'techstartup_lora.pt')

# Mit vollständiger Modellgröße vergleichen
torch.save(model.state_dict(), 'full_model.pt')
full_size = os.path.getsize('full_model.pt') / (1024 * 1024)
print(f"Vollständiges Modell gespeichert: full_model.pt ({full_size:.1f} MB)")

print(f"\nLoRA ist {full_size*1024/lora_size:.0f}x kleiner als das vollständige Modell!")

In [ ]:
# Laden demonstrieren
print("\n===== Laden/Speichern testen =====")

# Frisches Modell erstellen
new_model = MiniGPT(config).to(device)
new_model = add_lora_to_model(new_model, rank=8, alpha=16)

# Vor dem Laden testen
before_load = generate_response(new_model, "Was macht TechStartup Inc?", tokenizer)
print(f"Vor dem Laden von LoRA: {before_load[:60]}...")

# Gewichte laden
load_lora_weights(new_model, 'techstartup_lora.pt')

# Nach dem Laden testen
after_load = generate_response(new_model, "Was macht TechStartup Inc?", tokenizer)
print(f"Nach dem Laden von LoRA: {after_load}")

print("\nLoRA-Adapter erfolgreich gespeichert und geladen!")

## Zusammenfassung

**Was wir erstellt haben:**

1. **Instruktions-/Antwort-Datensatz** mit Loss-Maskierung
2. **LoRA-Adapter**, die <1% der Parameter trainieren
3. **Feinabstimmungs-Loop** mit dem 5-Schritte-Rezept
4. **Auswertung**, die dramatische Vorher-Nachher-Verbesserung zeigt
5. **Adapter-Speichern/Laden** für Deployment

**Wichtige Erkenntnisse:**

- Loss-Maskierung fokussiert das Training auf das Wesentliche (Antworten, nicht Instruktionen)
- LoRA erreicht großartige Ergebnisse mit 48× weniger trainierbaren Parametern
- Das Einfrieren der Basisgewichte verhindert **katastrophales Vergessen** (Verlust von vortrainiertem Wissen)
- Die Qualität der Daten ist wichtiger als die Quantität für die Feinabstimmung
- Adapter sind winzig und können für verschiedene Aufgaben ausgetauscht werden

**Nächstes:** Kapitel 14 wird Prompt Engineering erkunden - bessere Ausgaben erhalten, ohne die Modellgewichte überhaupt zu ändern!

## Übungen

### Übung 1: Verschiedene LoRA-Ränge

Probieren Sie rank=4 und rank=16 aus. Wie beeinflusst es Training und Ergebnisse?

In [ ]:
# IHR CODE HIER
# 1. Modell mit rank=4 erstellen
# 2. Für 3 Epochen trainieren
# 3. Loss und Auswertung mit rank=8 vergleichen

### Übung 2: Eigene FAQ erstellen

Erstellen Sie einen Datensatz über ein Thema, das Sie kennen (Ihre Schule, Hobby, etc.).

In [ ]:
# IHR CODE HIER
# 1. 30+ Instruktions-/Antwort-Paare erstellen
# 2. Modell feinabstimmen
# 3. Mit eigenen Fragen testen

### Übung 3: Loss-Maskierungs-Ablation

Was passiert, wenn wir die Instruktions-Tokens NICHT maskieren?

In [ ]:
# IHR CODE HIER
# 1. Dataset modifizieren, um Instruktions-Tokens NICHT zu maskieren (alle labels = tatsächliche Tokens)
# 2. Modell trainieren
# 3. Ergebnisse mit maskierter Version vergleichen
# 4. Was beobachten Sie?